In [1]:
# # Install dalex if not already installed
# # Run this once then comment it out
# import subprocess
# subprocess.run(['pip', 'install', 'dalex'], capture_output=True)

CompletedProcess(args=['pip', 'install', 'dalex'], returncode=0, stdout=b'Collecting dalex\r\n  Downloading dalex-1.8.0-py3-none-any.whl.metadata (30 kB)\r\nRequirement already satisfied: setuptools in c:\\users\\hp\\anaconda3\\lib\\site-packages (from dalex) (78.1.1)\r\nRequirement already satisfied: packaging in c:\\users\\hp\\anaconda3\\lib\\site-packages (from dalex) (24.1)\r\nRequirement already satisfied: pandas<3.0.0,>=1.5.3 in c:\\users\\hp\\anaconda3\\lib\\site-packages (from dalex) (2.3.3)\r\nRequirement already satisfied: numpy>=1.23.5 in c:\\users\\hp\\anaconda3\\lib\\site-packages (from dalex) (1.26.4)\r\nRequirement already satisfied: scipy>=1.6.3 in c:\\users\\hp\\anaconda3\\lib\\site-packages (from dalex) (1.13.1)\r\nCollecting plotly>=6.0.0 (from dalex)\r\n  Downloading plotly-6.6.0-py3-none-any.whl.metadata (8.5 kB)\r\nRequirement already satisfied: tqdm>=4.61.2 in c:\\users\\hp\\anaconda3\\lib\\site-packages (from dalex) (4.66.5)\r\nRequirement already satisfied: pyt

In [3]:
import dalex as dx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print(f"Dalex version: {dx.__version__}")
print("All imports successful.")

Dalex version: 1.8.0
All imports successful.


## Notebook 6: Dalex Model Explainability

### What Dalex adds that SHAP does not cover

SHAP (Notebook 4) answered: why did the model score 
THIS SPECIFIC borrower a certain way?

Dalex answers different questions:
1. **Variable Importance** — which features drive model 
   performance across ALL borrowers, measured by how much 
   performance drops when each feature is removed?

2. **Partial Dependence Profiles** — what is the functional 
   relationship between each feature and default risk across 
   the full range of values in the population?

3. **Residual Analysis** — where does the model get it wrong? 
   Which borrower profiles are systematically misjudged?

4. **Model Comparison** — how does XGBoost behave differently 
   from Logistic Regression across the same population?

### Why this matters for Nigerian credit risk
Regulators and auditors do not ask about individual decisions.
They ask about model behaviour at a portfolio level.
Dalex provides the portfolio-level view.

In [6]:
# Load clean dataset
df = pd.read_csv(r"C:\Users\HP\Documents\ML Project\data\processed\application_train_clean.csv")
X = df.drop(columns=['TARGET'])
y = df['TARGET']

# Same split as all previous notebooks
# random_state=42 guarantees identical test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Default rate: {y.mean()*100:.2f}%")

# Load saved XGBoost model
import joblib
import os

base_dir = os.path.dirname(os.path.abspath('__file__'))
models_dir = os.path.join('..', 'models')

xgb_model = joblib.load(os.path.join(models_dir, r"C:\Users\HP\Documents\ML Project\loan-default-prediction-\models\xgb_model.pkl"))
print(f"\nXGBoost model loaded successfully.")
print(f"Model type: {type(xgb_model)}")

Train: 246,008 | Test: 61,503
Default rate: 8.07%

XGBoost model loaded successfully.
Model type: <class 'xgboost.sklearn.XGBClassifier'>


In [8]:
# Recreate the Logistic Regression baseline for comparison
# Dalex's model comparison is one of its strongest features
# We need both models in the same framework

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    C=0.1
)
lr_model.fit(X_train_scaled, y_train)

print("Logistic Regression retrained.")
print("Both models ready for Dalex comparison.")

Logistic Regression retrained.
Both models ready for Dalex comparison.


In [10]:
# An Explainer is Dalex's core object
# Think of it as a wrapper that connects:
# - your model
# - your data  
# - your labels
# - a human-readable name
# Once created, all Dalex analysis flows from this object

print("Creating Dalex explainer for XGBoost...")
explainer_xgb = dx.Explainer(
    model=xgb_model,
    data=X_test,
    y=y_test,
    label="XGBoost",
    verbose=False
)

print("Creating Dalex explainer for Logistic Regression...")
explainer_lr = dx.Explainer(
    model=lr_model,
    data=pd.DataFrame(
        X_test_scaled, 
        columns=X_test.columns
    ),
    y=y_test,
    label="Logistic Regression",
    verbose=False
)

print("\nBoth explainers created successfully.")
print("Dalex is now connected to both models.")
print("All subsequent analysis runs through these explainer objects.")

Creating Dalex explainer for XGBoost...
Creating Dalex explainer for Logistic Regression...

Both explainers created successfully.
Dalex is now connected to both models.
All subsequent analysis runs through these explainer objects.


In [14]:
print("Calculating variable importance...")
print("This runs the model thousands of times on shuffled data.")
print("Takes 3-5 minutes. Do not interrupt.\n")

# Calculate permutation-based variable importance
# for both models simultaneously
vi_xgb = explainer_xgb.model_parts(
    loss_function='auc',
    N=2000,           
    B=10,             
    random_state=42
)

vi_lr = explainer_lr.model_parts(
    loss_function='auc',
    N=2000,
    B=10,
    random_state=42
)

print("Variable importance calculated.")

Calculating variable importance...
This runs the model thousands of times on shuffled data.
Takes 3-5 minutes. Do not interrupt.

Variable importance calculated.


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# XGBoost importance
xgb_imp = vi_xgb.result
xgb_imp = xgb_imp[
    xgb_imp['variable'] != '_baseline_'
].nlargest(15, 'dropout_loss')

axes[0].barh(
    xgb_imp['variable'][::-1],
    xgb_imp['dropout_loss'][::-1],
    color='#2E75B6',
    edgecolor='none',
    height=0.6
)
axes[0].set_title(
    'XGBoost — Variable Importance\n'
    '(Permutation-based: drop in AUC when feature is shuffled)',
    fontweight='bold', fontsize=11
)
axes[0].set_xlabel('Drop in AUC-ROC (higher = more important)')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Logistic Regression importance
lr_imp = vi_lr.result
lr_imp = lr_imp[
    lr_imp['variable'] != '_baseline_'
].nlargest(15, 'dropout_loss')

axes[1].barh(
    lr_imp['variable'][::-1],
    lr_imp['dropout_loss'][::-1],
    color='#888780',
    edgecolor='none',
    height=0.6
)
axes[1].set_title(
    'Logistic Regression — Variable Importance\n'
    '(Permutation-based: drop in AUC when feature is shuffled)',
    fontweight='bold', fontsize=11
)
axes[1].set_xlabel('Drop in AUC-ROC (higher = more important)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle(
    'Variable Importance Comparison — XGBoost vs Logistic Regression\n'
    'Home Credit Loan Default Dataset',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(
    '../reports/dalex_variable_importance.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print("Chart saved.")

Chart saved.


In [20]:
print("Calculating partial dependence profiles...")
print("Shows how default probability changes across feature ranges.")
print("Takes 2-3 minutes.\n")

# Select top 4 features for partial dependence
top_features = [
    'EXT_SOURCE_MEAN',
    'CREDIT_INCOME_RATIO', 
    'ANNUITY_INCOME_RATIO',
    'AGE_YEARS'
]

# Calculate profiles for both models
pdp_xgb = explainer_xgb.model_profile(
    variables=top_features,
    type='partial',
    N=1000,
    random_state=42
)

pdp_lr = explainer_lr.model_profile(
    variables=top_features,
    type='partial',
    N=1000,
    random_state=42
)

print("Profiles calculated.")

Calculating partial dependence profiles...
Shows how default probability changes across feature ranges.
Takes 2-3 minutes.



Calculating ceteris paribus: 100%|███████████████████████████████████████████████████████| 4/4 [00:00<00:00,  5.40it/s]


Profiles calculated.


In [22]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

feature_labels = {
    'EXT_SOURCE_MEAN': 'Credit Bureau Score (Average)',
    'CREDIT_INCOME_RATIO': 'Loan-to-Income Ratio',
    'ANNUITY_INCOME_RATIO': 'Monthly Repayment Burden',
    'AGE_YEARS': 'Borrower Age (Years)'
}

for i, feature in enumerate(top_features):
    
    # XGBoost profile
    xgb_data = pdp_xgb.result[
        pdp_xgb.result['_vname_'] == feature
    ]
    axes[i].plot(
        xgb_data['_x_'],
        xgb_data['_yhat_'],
        color='#2E75B6',
        linewidth=2.5,
        label='XGBoost'
    )
    
    # Logistic Regression profile
    lr_data = pdp_lr.result[
        pdp_lr.result['_vname_'] == feature
    ]
    axes[i].plot(
        lr_data['_x_'],
        lr_data['_yhat_'],
        color='#888780',
        linewidth=2,
        linestyle='--',
        label='Logistic Regression'
    )
    
    axes[i].set_title(
        feature_labels[feature],
        fontweight='bold',
        fontsize=11
    )
    axes[i].set_xlabel('Feature Value')
    axes[i].set_ylabel('Predicted Default Probability')
    axes[i].legend(fontsize=9)
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)
    axes[i].grid(axis='y', alpha=0.3)

plt.suptitle(
    'Partial Dependence Profiles — How Each Feature Affects Default Risk\n'
    'XGBoost vs Logistic Regression · Home Credit Dataset',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(
    '../reports/dalex_partial_dependence.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print("Chart saved.")

Chart saved.


In [26]:
print("Calculating model residuals...")

# In newer versions of Dalex, residuals are accessed
# through model_performance().result not .residuals directly

perf_xgb = explainer_xgb.model_performance(
    model_type='classification'
)
perf_lr = explainer_lr.model_performance(
    model_type='classification'
)

# Get predictions and calculate residuals manually
# This works across all Dalex versions
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
lr_probs = lr_model.predict_proba(
    pd.DataFrame(X_test_scaled, columns=X_test.columns)
)[:, 1]

# Residual = actual outcome - predicted probability
xgb_residuals = y_test.values - xgb_probs
lr_residuals = y_test.values - lr_probs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XGBoost residuals
axes[0].hist(
    xgb_residuals,
    bins=50,
    color='#2E75B6',
    edgecolor='none',
    alpha=0.8
)
axes[0].axvline(
    x=0, color='#E24B4A',
    linewidth=1.5,
    linestyle='--',
    label='Perfect prediction'
)
axes[0].set_title(
    'XGBoost — Residual Distribution\n'
    '(residual = actual − predicted probability)',
    fontweight='bold'
)
axes[0].set_xlabel('Residual Value')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Logistic Regression residuals
axes[1].hist(
    lr_residuals,
    bins=50,
    color='#888780',
    edgecolor='none',
    alpha=0.8
)
axes[1].axvline(
    x=0, color='#E24B4A',
    linewidth=1.5,
    linestyle='--',
    label='Perfect prediction'
)
axes[1].set_title(
    'Logistic Regression — Residual Distribution\n'
    '(residual = actual − predicted probability)',
    fontweight='bold'
)
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle(
    'Residual Analysis — Where Does Each Model Get It Wrong?\n'
    'Negative residuals = model over-predicted risk  |  '
    'Positive residuals = model under-predicted risk',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig(
    '../reports/dalex_residuals.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

print("\nResidual interpretation:")
print(f"XGBoost mean residual:             "
      f"{xgb_residuals.mean():.4f}")
print(f"Logistic Regression mean residual: "
      f"{lr_residuals.mean():.4f}")
print(f"\nXGBoost residual std:              "
      f"{xgb_residuals.std():.4f}")
print(f"Logistic Regression residual std:  "
      f"{lr_residuals.std():.4f}")
print("\nCloser to 0 mean = better calibrated")
print("Smaller std = tighter, more consistent predictions")

Calculating model residuals...

Residual interpretation:
XGBoost mean residual:             -0.3063
Logistic Regression mean residual: -0.3410

XGBoost residual std:              0.2944
Logistic Regression residual std:  0.2935

Closer to 0 mean = better calibrated
Smaller std = tighter, more consistent predictions


In [28]:
# Pick a high-risk borrower from the test set
# and explain their specific prediction using Dalex
# This complements the SHAP waterfall from Notebook 4

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
high_risk_idx = np.argmax(xgb_probs)

print(f"Analysing highest-risk borrower:")
print(f"Index: {high_risk_idx}")
print(f"Predicted default probability: "
      f"{xgb_probs[high_risk_idx]*100:.1f}%")
print(f"Actual outcome: "
      f"{'Defaulted' if y_test.iloc[high_risk_idx]==1 else 'Did not default'}")

# Dalex prediction breakdown
# Similar to SHAP waterfall but model-agnostic
breakdown = explainer_xgb.predict_parts(
    new_observation=X_test.iloc[[high_risk_idx]],
    type='break_down',
    order=None
)

# Plot the breakdown
bd_result = breakdown.result
bd_result = bd_result[
    bd_result['variable'] != 'intercept'
].copy()
bd_result = bd_result.reindex(
    bd_result['contribution'].abs().sort_values(ascending=False).index
).head(12)

colors = [
    '#E24B4A' if v > 0 else '#2E75B6' 
    for v in bd_result['contribution']
]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    bd_result['variable'][::-1],
    bd_result['contribution'][::-1],
    color=colors[::-1],
    edgecolor='none',
    height=0.6
)
ax.axvline(x=0, color='#333', linewidth=0.8)
ax.set_xlabel('Contribution to Default Probability', fontsize=11)
ax.set_title(
    f'Dalex Prediction Breakdown — High Risk Borrower\n'
    f'Predicted probability: {xgb_probs[high_risk_idx]*100:.1f}% default risk\n'
    f'Red = increases risk  |  Blue = reduces risk',
    fontweight='bold', fontsize=11, pad=12
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(
    '../reports/dalex_breakdown.png',
    dpi=150, bbox_inches='tight'
)
plt.show()
print("Dalex breakdown chart saved.")

Analysing highest-risk borrower:
Index: 4981
Predicted default probability: 95.2%
Actual outcome: Defaulted
Dalex breakdown chart saved.


In [30]:
print("=" * 60)
print("DALEX EXPLAINABILITY SUMMARY")
print("Notebook 6 — Home Credit Loan Default Project")
print("=" * 60)
print(f"""
ANALYSES COMPLETED:

1. Variable Importance (Permutation-based)
   → Which features drive model performance across all borrowers?
   → Saved: reports/dalex_variable_importance.png

2. Partial Dependence Profiles
   → How does default risk change across feature ranges?
   → Top 4 features: EXT_SOURCE_MEAN, CREDIT_INCOME_RATIO,
     ANNUITY_INCOME_RATIO, AGE_YEARS
   → Saved: reports/dalex_partial_dependence.png

3. Residual Analysis
   → Where does each model systematically get it wrong?
   → XGBoost vs Logistic Regression comparison
   → Saved: reports/dalex_residuals.png

4. Prediction Breakdown
   → Why did the model score the highest-risk borrower this way?
   → Dalex breakdown complements SHAP waterfall from Notebook 4
   → Saved: reports/dalex_breakdown.png

DALEX vs SHAP — KEY DIFFERENCE:
SHAP answers: why did THIS borrower get THIS score?
Dalex answers: how does the model behave across ALL borrowers?
Both are needed for a complete picture.

COMPLETE PROJECT NOTEBOOK SUMMARY:
01_eda.ipynb              → Data exploration
02_feature_engineering    → Domain-driven features
03_baseline_model         → Logistic Regression (AUC: 0.7484)
04_xgboost_shap           → XGBoost + SHAP (AUC: 0.7679)
05_anomaly_detection      → Isolation Forest + One-Class SVM
06_dalex_explainability   → Portfolio-level model explanation
""")
print("=" * 60)

DALEX EXPLAINABILITY SUMMARY
Notebook 6 — Home Credit Loan Default Project

ANALYSES COMPLETED:

1. Variable Importance (Permutation-based)
   → Which features drive model performance across all borrowers?
   → Saved: reports/dalex_variable_importance.png

2. Partial Dependence Profiles
   → How does default risk change across feature ranges?
   → Top 4 features: EXT_SOURCE_MEAN, CREDIT_INCOME_RATIO,
     ANNUITY_INCOME_RATIO, AGE_YEARS
   → Saved: reports/dalex_partial_dependence.png

3. Residual Analysis
   → Where does each model systematically get it wrong?
   → XGBoost vs Logistic Regression comparison
   → Saved: reports/dalex_residuals.png

4. Prediction Breakdown
   → Why did the model score the highest-risk borrower this way?
   → Dalex breakdown complements SHAP waterfall from Notebook 4
   → Saved: reports/dalex_breakdown.png

DALEX vs SHAP — KEY DIFFERENCE:
SHAP answers: why did THIS borrower get THIS score?
Dalex answers: how does the model behave across ALL borrowers?
Bot